## PRACTICA DEEP LEARNING: FERNANDO AMALFITANO

In [20]:
# Importamos las librerías que vamos a utilizar en el proyecto:
import os
import zipfile
import pandas as pd
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

In [5]:
# Cargado de zip data_main a la carpeta /content, en este archivo tenemos las
# imagenes que utilizaremos en el proyecto:
uploaded = files.upload()

with zipfile.ZipFile("data_main.zip", "r") as zip_ref:
    zip_ref.extractall("/content")

Saving data_main.zip to data_main.zip


In [7]:
# Cargado del archivo csv con la metadata a la carpeta /content:
from google.colab import files

uploaded = files.upload()

Saving poi_dataset.csv to poi_dataset (3).csv


In [9]:
# Carga de csv al notebook:
df = pd.read_csv("/content/poi_dataset.csv")
df.head()

,id,name,shortDescription,categories,tier,locationLon,locationLat,tags,xps,Visits,Likes,Dislikes,Bookmarks,main_image_path
0,4b36a3ed-3b28-4bc7-b975-1d48b586db03,Galería Fran Reus,La Galería Fran Reus es un espacio dedicado a ...,"['Escultura', 'Pintura']",1,2.642262,39.572694,[],500,10009,422,3582,78,data_main/4b36a3ed-3b28-4bc7-b975-1d48b586db03...
1,e32b3603-a94f-49df-8b31-92445a86377c,Convento de San Plácido,"El Convento de San Plácido en Madrid, fundado ...","['Patrimonio', 'Historia']",1,-3.704467,40.423037,[],500,10010,7743,96,2786,data_main/e32b3603-a94f-49df-8b31-92445a86377c...
2,0123a69b-13ac-4b65-a5d5-71a95560cff5,Instituto Geológico y Minero de España,"El Instituto Geológico y Minero de España, sit...","['Ciencia', 'Patrimonio']",2,-3.699694,40.442045,[],250,10015,3154,874,595,data_main/0123a69b-13ac-4b65-a5d5-71a95560cff5...
3,390d7d9e-e972-451c-b5e4-f494af15e788,Margarita Gil Roësset,"Margarita Gil Roësset, escultora y poetisa esp...",['Cultura'],1,-3.691228,40.427256,[],500,10011,8559,79,2358,data_main/390d7d9e-e972-451c-b5e4-f494af15e788...
4,023fc1bf-a1cd-4b9f-af78-48792ab1a294,Museo del Traje. Centro de Investigación del P...,"El Museo del Traje de Madrid, fundado en 2004,...","['Patrimonio', 'Cultura']",1,-3.727822,40.439665,[],500,10020,915,2896,143,data_main/023fc1bf-a1cd-4b9f-af78-48792ab1a294...


In [11]:

# Se crea una nueva columna con la ruta completa de cada
# imagen para que posteriormente pueda ser cargada por
# la red neuronal convolucional.

df["image_path"] = df["main_image_path"].apply(
    lambda x: f"/content/{x}"
)

In [12]:
# Comprobación de que la ruta existe:
print(df["image_path"].iloc[0])
print(os.path.exists(df["image_path"].iloc[0]))

/content/data_main/4b36a3ed-3b28-4bc7-b975-1d48b586db03/main.jpg
True


In [13]:
# Creación de la variable objetivo:

# Las métricas de interacción tienen escalas diferentes.
# Para hacerlas comparables se normalizan entre 0 y 1
# utilizando MinMaxScaler

# Pondero las métricas por diferentes valores dependiendo del
# peso que tienen estas bajo mi criterio. Sumo las que aportan
# valor al resultado final y resto "Dislikes" ya que este valor
# penaliza el engagement.

metrics = ["Likes", "Bookmarks", "Visits", "Dislikes"]

scaler = MinMaxScaler()

df[[m + "_n" for m in metrics]] = scaler.fit_transform(
    df[metrics]
)

df["engagement_score"] = (
    0.4 * df["Likes_n"]
    + 0.3 * df["Bookmarks_n"]
    + 0.2 * df["Visits_n"]
    - 0.1 * df["Dislikes_n"]
)

# Por ultimo genero la etiqueta binaria
# y = 1 -> engagement superior a la mediana
# y = 0 -> engagement inferior o igual a la mediana

df["y"] = (
    df["engagement_score"]
    > df["engagement_score"].median()
).astype(int)

In [14]:
# División del dataset

# Divido el conjunto de datos:
# 80% Train
# 20% Test
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["y"],
    random_state=42
)

In [15]:
# Si existe una GPU disponible se utilizará para acelerar
# el entrenamiento. En caso contrario se utilizará la CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cpu


In [16]:
# Preprocesamiento de imágenes:

# Las imágenes se redimensionan a 224x224 píxeles,
# tamaño requerido por EfficientNet-B0.

# Posteriormente se convierten a tensores para que puedan
# ser procesadas por PyTorch.

img_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [17]:
# Extracción de características visuales:

# Se utiliza EfficientNet-B0 preentrenada en ImageNet.

# Se elimina la capa clasificadora final para emplear
# únicamente el extractor de características visuales.

cnn = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

cnn.classifier = nn.Identity()

cnn = cnn.to(device)

cnn.eval()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 365MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [18]:
# Se elimina la capa clasificadora final para emplear
# únicamente el extractor de características visuales.

# El modelo actuará únicamente como extractor de
# características visuales.

for param in cnn.parameters():
    param.requires_grad = False

In [24]:
# Red neuronal para metadatos

# Esta red procesa la información tabular:
# - tier
# - locationLon
# - locationLat
# - xps

meta_net = nn.Sequential(
    nn.Linear(4, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU()
).to(device)

In [25]:
# Clasificador multimodal

# Aquí se fusionan las características visuales y los metadatos,
# para realizar la clasificación final:

clf = nn.Sequential(
    nn.Linear(1280 + 16, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
    nn.Sigmoid()
).to(device)

In [26]:
# Configuración del entrenamiento:

# Utilizo BCELoss para clasificación binaria y Adam como
# algoritmo de optimización

criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    list(meta_net.parameters()) +
    list(clf.parameters()),
    lr=1e-4
)

In [27]:
# Fijo semillas aleatorias para que los resultados
# puedan reproducirse en futuras ejecuciones.

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [28]:
# Entrenamiento del modelo

# Para cada muestra:

# Se extraen características visuales mediante
# EfficientNet.

# Se procesan los metadatos mediante una red neuronal.

# Se fusionan ambas representaciones.

# Se calcula la predicción.

# Se calcula la pérdida y se actualizan los pesos.


epochs = 20

for epoch in range(epochs):

    total_loss = 0


    meta_net.train()
    clf.train()

    for _, row in train_df.iterrows():

        # Imagen

        img = Image.open(
            row["image_path"]
        ).convert("RGB")

        img = img_tf(img).unsqueeze(0).to(device)

        with torch.no_grad():
            img_feat = cnn(img).squeeze()

        # Metadatos

        meta = torch.tensor([
            row["tier"],
            row["locationLon"],
            row["locationLat"],
            row["xps"]
        ], dtype=torch.float32).to(device)

        meta_feat = meta_net(meta)

        # Fusión

        x = torch.cat(
            [img_feat, meta_feat],
            dim=-1
        )

        out = clf(x)

        y = torch.tensor(
            row["y"],
            dtype=torch.float32
        ).to(device)

        loss = criterion(
            out.squeeze(),
            y
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1} - Loss: "
        f"{total_loss/len(train_df):.4f}"
    )

Epoch 1 - Loss: 0.5865
Epoch 2 - Loss: 0.4971
Epoch 3 - Loss: 0.4650
Epoch 4 - Loss: 0.4427
Epoch 5 - Loss: 0.4246
Epoch 6 - Loss: 0.4082
Epoch 7 - Loss: 0.3937
Epoch 8 - Loss: 0.3788
Epoch 9 - Loss: 0.3638
Epoch 10 - Loss: 0.3492
Epoch 11 - Loss: 0.3345
Epoch 12 - Loss: 0.3199
Epoch 13 - Loss: 0.3056
Epoch 14 - Loss: 0.2917
Epoch 15 - Loss: 0.2771
Epoch 16 - Loss: 0.2634
Epoch 17 - Loss: 0.2496
Epoch 18 - Loss: 0.2367
Epoch 19 - Loss: 0.2246
Epoch 20 - Loss: 0.2124


In [30]:
# Evaluación del modelo

# Se evalúa el rendimiento utilizando el conjunto de
# prueba, calculando:
# - Accuracy
# - Precision
# - Recall
# - F1-score

meta_net.eval()
clf.eval()

correct = 0

y_true = []
y_pred = []

with torch.no_grad():

    for _, row in test_df.iterrows():

        img = Image.open(
            row["image_path"]
        ).convert("RGB")

        img = img_tf(img).unsqueeze(0).to(device)

        img_feat = cnn(img).squeeze()

        meta = torch.tensor([
            row["tier"],
            row["locationLon"],
            row["locationLat"],
            row["xps"]
        ], dtype=torch.float32).to(device)

        meta_feat = meta_net(meta)

        x = torch.cat(
            [img_feat, meta_feat],
            dim=-1
        )

        pred = clf(x).item()

        pred_label = int(pred > 0.5)

        correct += (
            pred_label == row["y"]
        )

        y_true.append(row["y"])
        y_pred.append(pred_label)

accuracy = correct / len(test_df)

print("\nAccuracy:", accuracy)


Accuracy: 0.7388535031847133


In [31]:
# Evaluación mediante métricas de clasificación:

# Una vez entrenado el modelo, se comparan las etiquetas
# reales (y_true) con las predicciones obtenidas por el
# clasificador (y_pred).
print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred
    )
)


Classification Report:

              precision    recall  f1-score   support

           0       0.73      0.75      0.74       157
           1       0.75      0.73      0.74       157

    accuracy                           0.74       314
   macro avg       0.74      0.74      0.74       314
weighted avg       0.74      0.74      0.74       314



INTERPRETACIÓN DE LOS RESULTADOS:

Las métricas Precision, Recall y F1-Score presentan
valores muy similares para ambas clases, lo que indica
un comportamiento equilibrado y ausencia de sesgo
significativo hacia una clase concreta.

Estos resultados nos indican que la combinación de
características visuales extraídas mediante EfficientNet
y los metadatos del POI nos permite capturar información
relevante para predecir el nivel de engagement.




In [32]:
# Almacenamiento del modelo:

# Se guardan los pesos entrenados para poder reutilizar
# el modelo posteriormente sin necesidad de volver a
# entrenarlo.

torch.save(
    {
        "meta_net": meta_net.state_dict(),
        "classifier": clf.state_dict()
    },
    "modelo_multimodal.pth"
)

print("\nModelo guardado como modelo_multimodal.pth")


Modelo guardado como modelo_multimodal.pth
